# AIC 2026 — Kaggle Notebook A: GPU Preprocessing & Indexing

**Purpose**: Batch-encode keyframes using **MobileCLIP2-S4** on Kaggle T4x2 GPUs, build FAISS vector indices, and build BM25 raw text indices. Output artifacts are saved to `/kaggle/working/` to be attached as an input dataset for **Notebook B (Search)**.

**Model weights**: Attach the `mobileclip2-s4-weights` Kaggle dataset (containing `MobileCLIP2_S4.pt`) as an input to this notebook.

**Execution mode**: Committed run ("Save & Run All") with GPU accelerator (T4x2).

In [1]:
# Cell 1: Environment Setup & Package Installation
!pip install -q open_clip_torch faiss-gpu rank_bm25 tqdm pillow pandas numpy sentence-transformers


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.5/1.5 MB 21.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 135.2/135.2 MB 12.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 3.3 MB/s eta 0:00:00


In [2]:
# Cell 2: Imports and Constants
import os
import glob
import json
import time
import pickle
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor, as_completed

import torch
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm
import faiss
from rank_bm25 import BM25Okapi
import open_clip

# Find uploaded preprocessed data dataset dynamically
# Search recursively for the keyframe_csvs directory
DATASET_ROOT = Path("/kaggle/input/datasets/justavnesedude/aic2026-preprocessed-data")

if DATASET_ROOT is None:
    # Last resort: find any .csv file and infer root
    found = glob.glob("/kaggle/input/**/keyframe_csvs/*.csv", recursive=True)
    if found:
        DATASET_ROOT = Path(found[0]).parent.parent
        print(f"Found via recursive search: {DATASET_ROOT}")

print(f"DATASET_ROOT: {DATASET_ROOT}")
# Print actual input structure for debugging
print("Input dirs:", os.listdir("/kaggle/input"))
    

OUTPUT_DIR = Path("/kaggle/working")
CHECKPOINT_DIR = OUTPUT_DIR / "checkpoints"
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)

# MobileCLIP2-S4: loaded from the attached 'mobileclip2-s4-weights' Kaggle dataset
MODEL_NAME = "MobileCLIP2-S4"

import glob as _glob
MODEL_WEIGHTS_PATH = "/kaggle/input/models/justavnesedude/mobileclip2-s4/pytorch/default/1/mobileclip2_s4.pt"


# Reduced to 32 (from 64) for MobileCLIP2-S4 VRAM safety on T4
BATCH_SIZE = 32
CHECKPOINT_INTERVAL = 20  # Save checkpoint every N videos

print(f"Using DATASET_ROOT: {DATASET_ROOT}")
print(f"Using OUTPUT_DIR: {OUTPUT_DIR}")
print(f"Using MODEL: {MODEL_NAME} from {MODEL_WEIGHTS_PATH}")
print(f"PyTorch CUDA Available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU Count: {torch.cuda.device_count()}")
    for i in range(torch.cuda.device_count()):
        print(f"  cuda:{i} -> {torch.cuda.get_device_name(i)}")

DATASET_ROOT: /kaggle/input/datasets/justavnesedude/aic2026-preprocessed-data
Input dirs: ['datasets', 'models']
Using DATASET_ROOT: /kaggle/input/datasets/justavnesedude/aic2026-preprocessed-data
Using OUTPUT_DIR: /kaggle/working
Using MODEL: MobileCLIP2-S4 from /kaggle/input/models/justavnesedude/mobileclip2-s4/pytorch/default/1/mobileclip2_s4.pt
PyTorch CUDA Available: True
GPU Count: 2
  cuda:0 -> Tesla T4
  cuda:1 -> Tesla T4


In [3]:
# Cell 3: Metadata Assembly & Validation
print("Loading keyframe metadata CSVs...")
csv_files = sorted(glob.glob(str(DATASET_ROOT / "keyframe_csvs" / "*.csv")))
if not csv_files:
    csv_files = sorted(glob.glob(str(DATASET_ROOT / "**" / "keyframe_csvs" / "*.csv"), recursive=True))

print(f"Found {len(csv_files)} keyframe CSV files.")

meta_rows = []
dropped_count = 0
total_count = 0

for csv_path in csv_files:
    video_name = Path(csv_path).stem
    df_kf = pd.read_csv(csv_path)

    for _, row in df_kf.iterrows():
        total_count += 1
        img_name = f"{int(row['n']):04d}.jpg"
        img_path = DATASET_ROOT / "keyframe_images" / video_name / img_name
        if not img_path.exists():
            # Fast path check instead of slow rglob
            fallback = DATASET_ROOT / "L26_V100-V399" / "keyframe_images" / video_name / img_name
            if fallback.exists():
                img_path = fallback
            else:
                dropped_count += 1
                continue

        meta_rows.append({
            "video_name": video_name,
            "n": int(row['n']),
            "pts_time": float(row['pts_time']),
            "fps": float(row['fps']),
            "frame_idx": int(row['frame_idx']),
            "image_path": str(img_path)
        })

df_metadata = pd.DataFrame(meta_rows)

assert len(df_metadata) > 0, (
    f"0 keyframes found! DATASET_ROOT={DATASET_ROOT}. "
    f"Check that 'aic2026-preprocessed-data' dataset is attached."
)

print(f"Metadata assembly complete: {len(df_metadata)} valid keyframes.")
print(f"Dropped keyframes (missing files): {dropped_count} / {total_count} ({dropped_count/max(1,total_count):.2%})")

with open(OUTPUT_DIR / "metadata_df.pkl", "wb") as f:
    pickle.dump(df_metadata, f)

Loading keyframe metadata CSVs...
Found 300 keyframe CSV files.
Metadata assembly complete: 78044 valid keyframes.
Dropped keyframes (missing files): 0 / 78044 (0.00%)


In [4]:
# Cell 4: Dual-GPU MobileCLIP2-S4 Visual Encoding (Optimized with DataLoader)
class ChunkDataset(torch.utils.data.Dataset):
    def __init__(self, paths, preprocess):
        self.paths = paths
        self.preprocess = preprocess
    def __len__(self): 
        return len(self.paths)
    def __getitem__(self, idx):
        try:
            img = Image.open(self.paths[idx]).convert("RGB")
            return self.preprocess(img), idx
        except Exception:
            return torch.zeros(1), -1

def custom_collate(batch):
    batch = [b for b in batch if b[1] != -1]
    if not batch: return torch.empty(0), []
    tensors, indices = zip(*batch)
    return torch.stack(tensors), list(indices)

def worker_gpu_encode(gpu_id, chunk_df, model_name, weights_path):
    device = torch.device(f"cuda:{gpu_id}" if torch.cuda.is_available() else "cpu")
    # Load MobileCLIP2-S4 from local .pt file — call model.eval() for BatchNorm layers
    model, _, preprocess = open_clip.create_model_and_transforms(model_name, pretrained=weights_path)
    model = model.to(device).eval()

    paths = chunk_df["image_path"].tolist()
    dataset = ChunkDataset(paths, preprocess)
    
    # num_workers=4 offloads image loading & preprocessing to separate processes, bypassing GIL
    loader = torch.utils.data.DataLoader(
        dataset, 
        batch_size=BATCH_SIZE, 
        num_workers=4, 
        collate_fn=custom_collate,
        pin_memory=True
    )

    embeddings = []
    
    with torch.no_grad():
        for batch_tensors, _ in tqdm(loader, position=gpu_id, desc=f"GPU {gpu_id}"):
            if len(batch_tensors) == 0: continue
            batch_tensors = batch_tensors.to(device)
            feats = model.encode_image(batch_tensors)
            feats /= feats.norm(dim=-1, keepdim=True)
            embeddings.append(feats.cpu().numpy().astype(np.float32))

    if embeddings:
        return np.vstack(embeddings)
    return np.zeros((0, 512), dtype=np.float32)

# Checkpoint recovery
vis_emb_path = OUTPUT_DIR / "visual_embeddings.npy"
if vis_emb_path.exists():
    print("Loading existing visual_embeddings.npy from checkpoint...")
    visual_embeddings = np.load(vis_emb_path)
else:
    n_gpus = torch.cuda.device_count() if torch.cuda.is_available() else 1
    print(f"Starting MobileCLIP2-S4 encoding across {n_gpus} worker(s)...")

    chunks = np.array_split(df_metadata, n_gpus)
    start_t = time.time()

    if n_gpus > 1:
        with ThreadPoolExecutor(max_workers=n_gpus) as executor:
            futures = [executor.submit(worker_gpu_encode, i, chunks[i], MODEL_NAME, MODEL_WEIGHTS_PATH) for i in range(n_gpus)]
            results = [f.result() for f in futures]
        visual_embeddings = np.vstack(results)
    else:
        visual_embeddings = worker_gpu_encode(0, df_metadata, MODEL_NAME, MODEL_WEIGHTS_PATH)

    print(f"Encoding finished in {time.time() - start_t:.1f}s. Shape: {visual_embeddings.shape}")
    np.save(vis_emb_path, visual_embeddings)


Starting MobileCLIP2-S4 encoding across 2 worker(s)...


/usr/local/lib/python3.12/dist-packages/numpy/_core/fromnumeric.py:57: FutureWarning: 'DataFrame.swapaxes' is deprecated and will be removed in a future version. Please use 'DataFrame.transpose' instead.
  return bound(*args, **kwds)


Encoding finished in 2079.7s. Shape: (78044, 768)


In [5]:
# Cell 5: Build FAISS Visual Index
print("Building FAISS visual vector index (IndexFlatIP)...")
dim = visual_embeddings.shape[1]
visual_index = faiss.IndexFlatIP(dim)
visual_index.add(visual_embeddings)

faiss_path = OUTPUT_DIR / "visual_faiss.index"
faiss.write_index(visual_index, str(faiss_path))
print(f"Saved visual FAISS index: {visual_index.ntotal} vectors -> {faiss_path}")

Building FAISS visual vector index (IndexFlatIP)...
Saved visual FAISS index: 78044 vectors -> /kaggle/working/visual_faiss.index


In [6]:
# Cell 6: Caption & Subtitle Text Feature Encoding (BGE-m3 Text Encoder)
print("Loading captions & subtitles for text embeddings...")
text_documents = []

for idx, row in df_metadata.iterrows():
    v_name = row['video_name']
    pts = row['pts_time']

    sub_text = ""
    cap_text = ""

    # Quick path checks instead of slow rglob
    sub_file = DATASET_ROOT / "subtitles" / f"{v_name}.json"
    if not sub_file.exists():
        sub_file = DATASET_ROOT / "L26_V100-V399" / "subtitles" / f"{v_name}.json" # Fallback structure
    
    if sub_file.exists():
        try:
            with open(sub_file, "r", encoding="utf-8") as f:
                subs = json.load(f)
                sub_text = " ".join([s['text'] for s in subs if s['start'] <= pts <= s['end']])
        except Exception:
            pass

    cap_file = DATASET_ROOT / "captions" / f"{v_name}.json"
    if not cap_file.exists():
        cap_file = DATASET_ROOT / "L26_V100-V399" / "captions" / f"{v_name}.json"

    if cap_file.exists():
        try:
            with open(cap_file, "r", encoding="utf-8") as f:
                caps = json.load(f)
                for c in caps:
                    if c['start_time'] <= pts <= c['end_time']:
                        cap_text = c['caption']
                        break
        except Exception:
            pass

    combined = f"{sub_text} {cap_text}".strip()
    text_documents.append(combined if combined else "video scene")

print(f"Encoding {len(text_documents)} text documents with BGE-m3 text encoder...")
from sentence_transformers import SentenceTransformer
device = torch.device("cuda:0" if torch.cuda.is_available() else "cpu")
model_text = SentenceTransformer('BAAI/bge-m3').to(device)

# BGE-m3 already handles batching internally, but we can do it explicitly or pass the whole list
print("Extracting Text Features with BGE-m3...")
text_embeddings = model_text.encode(text_documents, batch_size=BATCH_SIZE, show_progress_bar=True, normalize_embeddings=True)
text_embeddings = text_embeddings.astype(np.float32)

np.save(OUTPUT_DIR / "text_embeddings.npy", text_embeddings)

dim_text = text_embeddings.shape[1]
text_index = faiss.IndexFlatIP(dim_text)
text_index.add(text_embeddings)
faiss.write_index(text_index, str(OUTPUT_DIR / "text_faiss.index"))
print(f"Saved text FAISS index: {text_index.ntotal} vectors of dimension {dim_text}")


Loading captions & subtitles for text embeddings...
Encoding 78044 text documents with MobileCLIP2-S4 text encoder...


Text Encoding:   0%|          | 0/2439 [00:00<?, ?it/s]

Saved text FAISS index: 78044 vectors


In [7]:
# Cell 7: Build BM25 Raw Text Corpus & Index
print("Building BM25 corpus (subtitles + captions + object detections)...")
bm25_corpus = []

for idx, row in df_metadata.iterrows():
    v_name = row['video_name']
    frame_idx = row['frame_idx']
    pts = row['pts_time']

    obj_text = ""
    # Direct checks instead of rglob
    obj_paths = [
        DATASET_ROOT / "objects" / v_name / f"{frame_idx:03d}.json",
        DATASET_ROOT / "objects" / v_name / f"{row['n']:04d}.json",
        DATASET_ROOT / "L26_V100-V399" / "objects" / v_name / f"{frame_idx:03d}.json",
    ]
    for opath in obj_paths:
        if opath.exists():
            try:
                with open(opath, "r", encoding="utf-8") as f:
                    o_data = json.load(f)
                    labels = o_data.get("detection_class_entities", [])
                    obj_text = " ".join(labels)
            except Exception:
                pass
            break

    doc_str = f"{text_documents[idx]} {obj_text}".strip()
    tokens = doc_str.lower().split()
    bm25_corpus.append(tokens if tokens else ["scene"])

print(f"Fitting BM25Okapi on {len(bm25_corpus)} documents...")
bm25 = BM25Okapi(bm25_corpus)

with open(OUTPUT_DIR / "bm25_corpus.pkl", "wb") as f:
    pickle.dump({"bm25": bm25, "corpus": bm25_corpus}, f)

print("BM25 index saved successfully.")

Building BM25 corpus (subtitles + captions + object detections)...
Fitting BM25Okapi on 78044 documents...
BM25 index saved successfully.


In [8]:
# Cell 8: Output Summary & Integrity Verification
print("="*60)
print("PREPROCESSING COMPLETE — ARTIFACT SUMMARY:")
print("="*60)
artifacts = [
    "metadata_df.pkl",
    "visual_embeddings.npy",
    "visual_faiss.index",
    "text_embeddings.npy",
    "text_faiss.index",
    "bm25_corpus.pkl"
]
for art in artifacts:
    p = OUTPUT_DIR / art
    if p.exists():
        size_mb = p.stat().st_size / (1024*1024)
        print(f"  [OK] {art:25s} -> {size_mb:8.2f} MB")
    else:
        print(f"  [MISSING] {art:25s}")
print("="*60)
print()
print("NEXT STEP: Download all 6 files above, put them in kaggle_output/")
print("in your local project, then run Notebook B locally.")

PREPROCESSING COMPLETE — ARTIFACT SUMMARY:
  [OK] metadata_df.pkl           ->    10.06 MB
  [OK] visual_embeddings.npy     ->   228.64 MB
  [OK] visual_faiss.index        ->   228.64 MB
  [OK] text_embeddings.npy       ->   228.64 MB
  [OK] text_faiss.index          ->   228.64 MB
  [OK] bm25_corpus.pkl           ->   104.21 MB

NEXT STEP: Download all 6 files above, put them in kaggle_output/
in your local project, then run Notebook B locally.
